# Validación completa del repositorio en Google Colab Free

**Objetivo:** clonar la rama privada, instalar dependencias de pruebas, ejecutar primero la prueba del balanceador que falló y después toda la suite `pytest`. Los logs se guardan en Google Drive incluso si hay fallos.

**No entrena modelos, no descarga datasets y no requiere GPU.** Mantén el entorno de ejecución en CPU; si Colab asigna GPU no se utilizará.

Antes de ejecutar: en Colab abre **Secrets** (icono de llave) y crea `GITHUB_TOKEN` con un Fine-grained PAT limitado a este repositorio y permiso `Contents: Read-only`. Alternativamente, la celda de clonación pedirá el token de forma oculta. Nunca lo pegues en el notebook ni en el chat.

## 1. Montar Drive para conservar los resultados

Autoriza Google Drive cuando aparezca la ventana. Los logs y el resumen se guardarán en `MyDrive/ProcesamientoImagenes/validacion_colab`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, textwrap, datetime

drive.mount('/content/drive')
RESULTS = Path('/content/drive/MyDrive/ProcesamientoImagenes/validacion_colab')
RESULTS.mkdir(parents=True, exist_ok=True)
print('Resultados persistentes:', RESULTS)


## 2. Obtener la versión actual del repositorio privado

La celda siempre actualiza a la última versión de `urban-models-and-benchmark`. El token solo vive en memoria durante el clonado.


In [ ]:
from getpass import getpass
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Leohf04/Procesamiento-de-Im-genes-.git'
BRANCH = 'urban-models-and-benchmark'
WORKDIR = Path('/content/Procesamiento-de-Im-genes-')
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass('GitHub Fine-grained token (Contents: Read-only; entrada oculta): ')
if not token:
    raise RuntimeError('No se proporcionó token de lectura del repositorio privado.')
askpass = Path('/tmp/colab_git_askpass.sh')
askpass.write_text('#!/bin/sh\\ncase "$1" in\\n  *Username*) echo x-access-token ;;\\n  *Password*) echo "$COLAB_GITHUB_TOKEN" ;;\\n  *) exit 1 ;;\\nesac\\n')
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'COLAB_GITHUB_TOKEN': token})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(WORKDIR)],check=True,env=env)
del token, env
os.chdir(WORKDIR)
HEAD = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Rama:', BRANCH)
print('HEAD:', HEAD)
print('Commit:', subprocess.check_output(['git','log','-1','--oneline'],text=True).strip())


## 3. Instalar dependencias necesarias para ejecutar las pruebas

No se reinstalan PyTorch ni Torchvision: se conserva la versión compatible que trae Colab. Esta fase no necesita CUDA.


In [ ]:
%cd /content/Procesamiento-de-Im-genes-
!python -m pip install -q 'pytest>=7' 'opencv-python-headless>=4.8,<5' 'scikit-learn>=1.3' 'matplotlib>=3.7' 'PyYAML>=6' 'tqdm>=4.66' 'joblib>=1.3' 'onnx>=1.14' 'onnxruntime>=1.17' 'Pillow>=9' 'pycocotools>=2.0.7' 'requests>=2.31'
print('Dependencias instaladas. No se modificaron torch/torchvision.')


## 4. Ejecutar las pruebas críticas y la suite completa

Primero se ejecuta expresamente la prueba que falló: `test_greedy_balancer_enforces_per_class_cap_and_can_reduce_training_set`. Después se ejecuta **toda la suite** aunque la prueba dirigida falle, para reunir el diagnóstico completo. Cada salida queda guardada en Drive.

La celda termina mostrando los códigos de salida. Si cualquier prueba falla, no se inicia entrenamiento ni se genera un plan experimental.


In [ ]:
import subprocess, sys, datetime, json
from pathlib import Path

os.chdir(WORKDIR)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
focused_log = RESULTS / f'prueba_balanceador_{stamp}.log'
full_log = RESULTS / f'suite_completa_{stamp}.log'
summary_path = RESULTS / f'resumen_pruebas_{stamp}.json'

focused_cmd = [sys.executable, '-m', 'pytest', '-q', 'tests/test_urban_ontology_and_balance.py::test_greedy_balancer_enforces_per_class_cap_and_can_reduce_training_set']
full_cmd = [sys.executable, '-m', 'pytest', '-q']

def run_and_save(command, log_path, label):
    print('\\n' + '='*80 + '\\n' + label + '\\n' + '='*80)
    proc = subprocess.run(command, cwd=WORKDIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path.write_text(proc.stdout, encoding='utf-8')
    print(proc.stdout[-18000:] if len(proc.stdout) > 18000 else proc.stdout)
    print('Código de salida:', proc.returncode)
    print('Log:', log_path)
    return proc.returncode

focused_rc = run_and_save(focused_cmd, focused_log, 'PRUEBA DIRIGIDA DEL BALANCEADOR')
full_rc = run_and_save(full_cmd, full_log, 'SUITE COMPLETA DEL REPOSITORIO')
summary = {
    'timestamp_local': datetime.datetime.now().astimezone().isoformat(),
    'repository': 'Leohf04/Procesamiento-de-Im-genes-',
    'branch': BRANCH,
    'git_head': HEAD,
    'focused_test_exit_code': focused_rc,
    'full_suite_exit_code': full_rc,
    'focused_log': str(focused_log),
    'full_log': str(full_log),
    'training_started': False,
    'datasets_downloaded': False,
    'passed': focused_rc == 0 and full_rc == 0,
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + '\\n', encoding='utf-8')
print('\\nRESUMEN FINAL')
print(json.dumps(summary, indent=2, ensure_ascii=False))
if not summary['passed']:
    print('\\nVALIDACIÓN NO SUPERADA: no continúes con balanceo ni entrenamiento. Comparte los dos logs guardados en Drive.')
else:
    print('\\nVALIDACIÓN SUPERADA: prueba dirigida y suite completa correctas. Aún no se ha entrenado ningún modelo.')


## 5. Qué enviarme al terminar

Si todo pasa, envíame una captura del bloque **RESUMEN FINAL**. Si algo falla, comparte el contenido de `prueba_balanceador_*.log` y `suite_completa_*.log` (o copia el error final). No ejecutes entrenamiento: el siguiente paso será inspeccionar el resultado y decidir si se continúa con la auditoría del plan de balanceo.

**Importante:** los logs permanecen en Google Drive aunque la sesión de Colab se desconecte. No contienen tu token GitHub.